# Thu Thập Dữ Liệu Google Trends (Interest Over Time, Region, Commonly Searched Queries)

Pipeline cào dữ liệu xu hướng từ Google Trends bằng `pytrends`:
- **Nguồn từ khóa**: Thuộc tính `mid` từ `data-raw/google-trends/trend_topics.csv`.
- **Tập mid duy nhất**: Chỉ truy vấn mỗi `mid` đúng 1 lần; nếu 1 `mid` ánh xạ nhiều `keyword` hoặc `keyword_type`, toàn bộ metadata liên quan sẽ được gắn kèm vào kết quả.
- **Các chỉ số thu thập**:
  1. `interest_over_time`: Mức độ quan tâm theo thời gian (chuỗi thời gian, trung bình).
  2. `interest_by_region`: Phân bố mức độ quan tâm theo tỉnh thành/khu vực.
  3. `commonly_searched_queries`: Gồm `top_queries` (phổ biến nhất) và `rising_queries` (đột biến/breakout).
- **Tham số truy vấn**: `geo='VI'`, `date='all'`.
- **Tối ưu hóa thời gian & độ tin cậy**:
  + Chạy song song nhiều truy vấn (`ThreadPoolExecutor`) với thread-safe session.
  + Xử lý Rate Limit (HTTP 429), ngắt kết nối và áp dụng Exponential Backoff with Jitter.
  + Lưu checkpoint định kỳ và hỗ trợ tiếp tục (resume) tiến trình mà không cào lại các mid đã hoàn tất.
- **Đầu ra**: Xuất toàn bộ dữ liệu vào một file JSON duy nhất (`data-raw/google-trends/trends.json`), là một array chứa các object kết quả của từng `mid`.

## 1. Khởi Tạo Môi Trường & Cấu Hình Tham Số

In [ ]:
# export
import argparse
import concurrent.futures
from datetime import datetime, timezone
import json
from pathlib import Path
import random
import sys
import threading
import time
from typing import Any, Dict, List, Optional, Tuple, Union

import pandas as pd
from pytrends.request import TrendReq

from helpers.get_configs import get_config

# Thiết lập UTF-8 output cho console trên Windows
if sys.platform == "win32" and hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

# Đọc cấu hình đường dẫn dự án từ config/dirs.yaml
TOPICS_PATH = get_config("dirs.yaml", ["raw_data", "trend_topics"], as_path=True)
TRENDS_PATH = get_config("dirs.yaml", ["raw_data", "trends"], as_path=True)
SCRIPT_OUTPUT_PATH = get_config("dirs.yaml", ["unit_scripts", "get_trends"], as_path=True)

# Tham số truy vấn Google Trends theo yêu cầu
HL = "vi-VN"
TZ = -420  # Múi giờ UTC+7 (phút)
GEO = "VI"  # Theo yêu cầu đề bài: geo='VI'
DATE = "all"  # Theo yêu cầu đề bài: date='all' (toàn bộ lịch sử từ 2004)

# Cấu hình retry & rate limiting
REQUEST_TIMEOUT = (10, 25)  # (connect, read) giây
MAX_RETRIES = 3  # Số lần thử lại tối đa cho mỗi mid
BACKOFF_BASE = 2.0  # Thời gian chờ cơ sở cho exponential backoff
SLEEP_RANGE = (1.5, 3.0)  # Nghỉ ngẫu nhiên giữa 2 request thành công trong cùng 1 luồng
CONSECUTIVE_FAIL_LIMIT = 5  # Số lần lỗi liên tiếp để kích hoạt cooldown
COOLDOWN_SECONDS = 60  # Nghỉ dài khi gặp chuỗi lỗi liên tiếp
MAX_WORKERS = 2  # Số luồng song song tối ưu để tránh bị Google chặn tần suất
SAVE_INTERVAL = 20  # Tần suất lưu checkpoint trung gian ra đĩa (số mid)

## 2. Đọc và Gom Nhóm Dữ Liệu Theo `mid` Duy Nhất

- Bỏ các dòng thiếu `mid` hoặc `mid` rỗng.
- Gom nhóm các dòng theo `mid` duy nhất.
- Nếu một `mid` ứng với nhiều `keyword` hoặc `keyword_type`, lưu danh sách ánh xạ vào `keyword_metadata` để chỉ cần truy vấn `mid` đó một lần duy nhất.

In [ ]:
# export
def load_and_group_topics(topics_path: Path) -> List[Dict[str, Any]]:
    """Đọc file trend_topics.csv và gom nhóm theo mid duy nhất kèm metadata đầy đủ."""
    df = pd.read_csv(topics_path, encoding="utf-8")
    valid_df = (
        df.dropna(subset=["mid"])
        .assign(mid=lambda d: d["mid"].astype(str).str.strip())
        .query("mid != ''")
        .reset_index(drop=True)
    )

    mid_dict: Dict[str, Dict[str, Any]] = {}
    for row in valid_df.itertuples(index=False):
        mid = getattr(row, "mid")
        kw = getattr(row, "keyword", "")
        kw_type = getattr(row, "keyword_type", "")
        title = getattr(row, "title", "")
        topic_type = getattr(row, "type", "")

        kw_clean = str(kw).strip() if pd.notna(kw) else ""
        type_clean = str(kw_type).strip() if pd.notna(kw_type) else ""

        if mid not in mid_dict:
            mid_dict[mid] = {
                "mid": mid,
                "title": str(title).strip() if pd.notna(title) else "",
                "type": str(topic_type).strip() if pd.notna(topic_type) else "",
                "keywords": [],
                "keyword_types": [],
                "keyword_metadata": [],
            }

        item = mid_dict[mid]
        if kw_clean and kw_clean not in item["keywords"]:
            item["keywords"].append(kw_clean)
        if type_clean and type_clean not in item["keyword_types"]:
            item["keyword_types"].append(type_clean)

        meta_pair = {"keyword": kw_clean, "keyword_type": type_clean}
        if meta_pair not in item["keyword_metadata"]:
            item["keyword_metadata"].append(meta_pair)

    tasks = []
    for mid, item in mid_dict.items():
        kws = item["keywords"]
        ktypes = item["keyword_types"]
        tasks.append({
            "mid": mid,
            "keyword": kws[0] if len(kws) == 1 else kws,
            "keyword_type": ktypes[0] if len(ktypes) == 1 else ktypes,
            "keyword_metadata": item["keyword_metadata"],
            "title": item["title"],
            "type": item["type"],
        })

    return tasks

## 3. Khởi Tạo Kết Nối Pytrends & Hàm Thu Thập Dữ Liệu Theo `mid`

- Sử dụng `threading.local` để đảm bảo an toàn đa luồng (Thread-Safety), mỗi worker thread sử dụng một instance `TrendReq` riêng biệt.
- Trích xuất đầy đủ 3 thành phần:
  1. `interest_over_time`
  2. `interest_by_region`
  3. `commonly_searched_queries` (Top queries & Rising queries)

In [ ]:
# export
_thread_local = threading.local()


def get_thread_trendreq() -> TrendReq:
    """Tạo hoặc lấy instance TrendReq riêng biệt cho từng worker thread."""
    if not hasattr(_thread_local, "pytrends") or _thread_local.pytrends is None:
        _thread_local.pytrends = TrendReq(
            hl=HL,
            tz=TZ,
            timeout=REQUEST_TIMEOUT,
        )
    return _thread_local.pytrends


def reset_thread_trendreq() -> None:
    """Làm mới kết nối TrendReq khi gặp lỗi Rate Limit 429."""
    if hasattr(_thread_local, "pytrends"):
        _thread_local.pytrends = None


def fetch_mid_data(
    mid_info: Dict[str, Any], geo: str = GEO, date: str = DATE
) -> Dict[str, Any]:
    """Truy vấn Google Trends cho 1 mid và bóc tách dữ liệu chuẩn hóa."""
    mid = mid_info["mid"]
    pt = get_thread_trendreq()

    # Xây dựng payload truy vấn
    pt.build_payload(kw_list=[mid], timeframe=date, geo=geo)

    # 1. Thu thập Interest over time
    timeline: List[Dict[str, Any]] = []
    avg_interest = 0.0
    try:
        iot = pt.interest_over_time()
        if iot is not None and not iot.empty and mid in iot.columns:
            for dt_idx, val in iot[mid].items():
                date_str = (
                    dt_idx.strftime("%Y-%m-%d")
                    if hasattr(dt_idx, "strftime")
                    else str(dt_idx)
                )
                timeline.append({"date": date_str, "value": int(val)})
            avg_interest = round(float(iot[mid].mean()), 2)
    except Exception:
        timeline = []
        avg_interest = 0.0

    # 2. Thu thập Interest by region
    regions: List[Dict[str, Any]] = []
    try:
        ibr = pt.interest_by_region()
        if ibr is not None and not ibr.empty and mid in ibr.columns:
            sorted_ibr = ibr.sort_values(by=mid, ascending=False)
            for rank, (reg_name, val) in enumerate(sorted_ibr[mid].items(), start=1):
                if int(val) > 0:
                    regions.append(
                        {"region": str(reg_name), "rank": rank, "value": int(val)}
                    )
    except Exception:
        regions = []

    # 3. Thu thập Commonly searched queries (Top queries & Rising queries)
    top_queries: List[Dict[str, Any]] = []
    rising_queries: List[Dict[str, Any]] = []
    try:
        rq = pt.related_queries() or {}
        mid_rq = rq.get(mid, {}) if isinstance(rq, dict) else {}

        top_df = mid_rq.get("top")
        if top_df is not None and not top_df.empty:
            for rank, row in enumerate(top_df.to_dict("records"), start=1):
                raw_val = row.get("value", 0)
                val = int(raw_val) if str(raw_val).isdigit() else raw_val
                top_queries.append({
                    "rank": rank,
                    "query": str(row.get("query", "")),
                    "value": val,
                })

        rising_df = mid_rq.get("rising")
        if rising_df is not None and not rising_df.empty:
            for rank, row in enumerate(rising_df.to_dict("records"), start=1):
                raw_val = row.get("value", "")
                val_str = str(raw_val).strip()
                is_breakout = val_str.lower() == "breakout"
                rising_queries.append({
                    "rank": rank,
                    "query": str(row.get("query", "")),
                    "value": raw_val,
                    "is_breakout": is_breakout,
                })
    except Exception:
        top_queries = []
        rising_queries = []

    return {
        "mid": mid,
        "keyword": mid_info["keyword"],
        "keyword_type": mid_info["keyword_type"],
        "keyword_metadata": mid_info["keyword_metadata"],
        "title": mid_info.get("title", ""),
        "type": mid_info.get("type", ""),
        "query_terms": [mid],
        "date_range": date,
        "geo": geo,
        "scraped_at": datetime.now(timezone.utc).isoformat(),
        "status": "success",
        "errors": [],
        "interest_over_time": {
            "timeline": timeline,
            "average_interest": avg_interest,
            "data_source": "pytrends",
        },
        "interest_by_region": {
            "regions": regions,
            "data_source": "pytrends",
        },
        "commonly_searched_queries": {
            "top_queries": top_queries,
            "rising_queries": rising_queries,
            "data_source": "pytrends",
        },
    }

## 4. Cơ Chế Thử Lại (Retry) và Exponential Backoff

- Tự động bắt lỗi HTTP 429 Too Many Requests và lỗi timeout mạng.
- Sử dụng công thức Exponential Backoff with Jitter: `wait = BACKOFF_BASE ** attempt + random(1, 3)`.
- Khi gặp 429: nhân đôi thời gian chờ và reset session kết nối của luồng.
- Nếu quá `MAX_RETRIES` lần thử vẫn thất bại: trả về object có `status='failed'` kèm thông tin lỗi để pipeline tiếp tục hoạt động.

In [ ]:
# export
def fetch_single_mid_with_retry(
    mid_info: Dict[str, Any],
    geo: str = GEO,
    date: str = DATE,
    max_retries: int = MAX_RETRIES,
) -> Dict[str, Any]:
    """Truy vấn 1 mid kèm cơ chế retry và exponential backoff."""
    mid = mid_info["mid"]
    last_error: Optional[Exception] = None

    for attempt in range(1, max_retries + 1):
        try:
            res = fetch_mid_data(mid_info, geo=geo, date=date)
            time.sleep(random.uniform(*SLEEP_RANGE))
            return res
        except Exception as e:
            last_error = e
            err_msg = str(e)
            is_rate_limit = "429" in err_msg or "Too Many Requests" in err_msg

            wait_time = (BACKOFF_BASE**attempt) + random.uniform(1.0, 3.0)
            if is_rate_limit:
                wait_time *= 2.0
                print(
                    f"  [429 Rate Limit] mid '{mid}' lần {attempt}/{max_retries} "
                    f"-> Chờ {wait_time:.1f}s và reset session..."
                )
                reset_thread_trendreq()
            else:
                print(
                    f"  [Lỗi mạng/API] mid '{mid}' lần {attempt}/{max_retries}: {e!r} "
                    f"-> Chờ {wait_time:.1f}s..."
                )

            time.sleep(wait_time)

    return {
        "mid": mid,
        "keyword": mid_info["keyword"],
        "keyword_type": mid_info["keyword_type"],
        "keyword_metadata": mid_info["keyword_metadata"],
        "title": mid_info.get("title", ""),
        "type": mid_info.get("type", ""),
        "query_terms": [mid],
        "date_range": date,
        "geo": geo,
        "scraped_at": datetime.now(timezone.utc).isoformat(),
        "status": "failed",
        "errors": [repr(last_error)],
        "interest_over_time": None,
        "interest_by_region": None,
        "commonly_searched_queries": None,
    }

## 5. Pipeline Điều Phối Song Song, Checkpoint & Lưu Trữ JSON Đầu Ra

In [ ]:
# export
def load_existing_records(file_path: Path) -> Tuple[List[Dict[str, Any]], set]:
    """Đọc file JSON đã cào trước đó để hỗ trợ resume/checkpoint."""
    if not file_path.exists() or file_path.stat().st_size == 0:
        return [], set()

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list):
                completed = {
                    item["mid"]
                    for item in data
                    if isinstance(item, dict)
                    and item.get("status") == "success"
                    and "mid" in item
                }
                return data, completed
    except Exception as e:
        print(f"  [warn] Không đọc được file checkpoint cũ: {e}")

    return [], set()


def save_trends_json(records: List[Dict[str, Any]], file_path: Path) -> None:
    """Ghi toàn bộ danh sách kết quả ra file JSON duy nhất."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = file_path.with_suffix(".tmp")
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    temp_path.replace(file_path)


def run_pipeline(
    topics_path: Path = TOPICS_PATH,
    output_path: Path = TRENDS_PATH,
    geo: str = GEO,
    date: str = DATE,
    max_workers: int = MAX_WORKERS,
    limit: Optional[int] = None,
    resume: bool = True,
) -> List[Dict[str, Any]]:
    """Điều phối quy trình cào Google Trends cho toàn bộ tập mid duy nhất."""
    print("=" * 70)
    print("BẮT ĐẦU QUY TRÌNH THU THẬP DỮ LIỆU GOOGLE TRENDS")
    print(f"- File nguồn: {topics_path}")
    print(f"- File đầu ra: {output_path}")
    print(f"- Cấu hình: geo='{geo}', date='{date}'")
    print(f"- Số luồng song song: {max_workers}")
    print("=" * 70)

    all_tasks = load_and_group_topics(topics_path)
    print(f"Tổng số mid duy nhất: {len(all_tasks)}")

    records: List[Dict[str, Any]] = []
    completed_mids = set()
    if resume:
        records, completed_mids = load_existing_records(output_path)
        if completed_mids:
            print(f"Đã tìm thấy checkpoint: {len(completed_mids)} mid đã hoàn tất thành công.")

    pending_tasks = [t for t in all_tasks if t["mid"] not in completed_mids]
    if limit is not None and limit > 0:
        pending_tasks = pending_tasks[:limit]
        print(f"Áp dụng limit={limit}: Số lượng mid xử lý trong đợt này là {len(pending_tasks)}.")
    else:
        print(f"Số lượng mid còn lại cần cào: {len(pending_tasks)}.")

    if not pending_tasks:
        print("Tất cả mid đã được thu thập hoàn tất!")
        return records

    record_map: Dict[str, Dict[str, Any]] = {r["mid"]: r for r in records if "mid" in r}
    lock = threading.Lock()
    processed_count = 0
    consecutive_fails = 0
    start_time = time.time()

    def worker_task(task_item: Dict[str, Any]) -> Dict[str, Any]:
        return fetch_single_mid_with_retry(task_item, geo=geo, date=date)

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_task = {
            executor.submit(worker_task, task): task for task in pending_tasks
        }

        for future in concurrent.futures.as_completed(future_to_task):
            task_info = future_to_task[future]
            mid = task_info["mid"]
            primary_kw = (
                task_info["keyword"]
                if isinstance(task_info["keyword"], str)
                else task_info["keyword"][0]
            )

            try:
                res = future.result()
            except Exception as e:
                res = {
                    "mid": mid,
                    "keyword": task_info["keyword"],
                    "keyword_type": task_info["keyword_type"],
                    "keyword_metadata": task_info["keyword_metadata"],
                    "title": task_info.get("title", ""),
                    "type": task_info.get("type", ""),
                    "query_terms": [mid],
                    "date_range": date,
                    "geo": geo,
                    "scraped_at": datetime.now(timezone.utc).isoformat(),
                    "status": "failed",
                    "errors": [repr(e)],
                    "interest_over_time": None,
                    "interest_by_region": None,
                    "commonly_searched_queries": None,
                }

            with lock:
                processed_count += 1
                record_map[mid] = res
                status = res.get("status", "unknown")

                if status == "failed":
                    consecutive_fails += 1
                else:
                    consecutive_fails = 0

                elapsed = time.time() - start_time
                avg_speed = processed_count / elapsed if elapsed > 0 else 0
                print(
                    f"[{processed_count}/{len(pending_tasks)}] "
                    f"({task_info['keyword_type']}) {primary_kw} | {mid} -> {status} "
                    f"({avg_speed:.2f} mid/s)"
                )

                if processed_count % SAVE_INTERVAL == 0:
                    save_trends_json(list(record_map.values()), output_path)
                    print(f"  --> [Checkpoint] Đã lưu {len(record_map)} bản ghi vào {output_path.name}")

                if consecutive_fails >= CONSECUTIVE_FAIL_LIMIT:
                    print(f"\n  [CẢNH BÁO] Thất bại liên tiếp {consecutive_fails} lần -> Tạm nghỉ {COOLDOWN_SECONDS}s...")
                    time.sleep(COOLDOWN_SECONDS)
                    consecutive_fails = 0

    final_records = list(record_map.values())
    save_trends_json(final_records, output_path)

    total_time = time.time() - start_time
    success_count = sum(1 for r in final_records if r.get("status") == "success")
    fail_count = sum(1 for r in final_records if r.get("status") == "failed")

    print("\n" + "=" * 70)
    print("HOÀN TẤT THU THẬP DỮ LIỆU GOOGLE TRENDS")
    print(f"- Tổng số đối tượng JSON: {len(final_records)}")
    print(f"- Thành công: {success_count} | Thất bại: {fail_count}")
    print(f"- Thời gian thực hiện: {total_time:.1f} giây")
    print(f"- File đã ghi: {output_path}")
    print("=" * 70)

    return final_records

## 6. Điểm Khởi Chạy Script (CLI Execution)

In [ ]:
# export
if False:  # Bản notebook cũ chỉ để tham khảo; không chạy pipeline này.
    parser = argparse.ArgumentParser(description="Thu thập dữ liệu Google Trends theo danh sách mid.")
    parser.add_argument("--limit", type=int, default=None, help="Giới hạn số mid cần thu thập (để test nhanh)")
    parser.add_argument("--workers", type=int, default=MAX_WORKERS, help="Số luồng song song (mặc định: 2)")
    parser.add_argument("--geo", type=str, default=GEO, help="Mã vùng địa lý Google Trends (mặc định: VI)")
    parser.add_argument("--date", type=str, default=DATE, help="Khoảng thời gian (mặc định: all)")
    parser.add_argument("--no-resume", action="store_true", help="Không dùng checkpoint, cào lại từ đầu")

    # Xử lý an toàn khi chạy từ môi trường interactive jupyter kernel
    if "ipykernel" in sys.modules:
        args = argparse.Namespace(limit=None, workers=MAX_WORKERS, geo=GEO, date=DATE, no_resume=False)
    else:
        args = parser.parse_args()

    run_pipeline(
        topics_path=TOPICS_PATH,
        output_path=TRENDS_PATH,
        geo=args.geo,
        date=args.date,
        max_workers=args.workers,
        limit=args.limit,
        resume=not args.no_resume,
    )

## 7. Pipeline Chính

Notebook này giữ lại để tham khảo lịch sử. Không export vì sẽ ghi đè pipeline cache mới tại `scripts-fetching/unit-scripts/get_trends.py`. Hãy chạy script theo hướng dẫn trong README.

In [ ]:
print("Pipeline chính: scripts-fetching/unit-scripts/get_trends.py")